In [ ]:
from google.colab import files
import zipfile
import os

print("Please upload your dataset ZIP file:")
uploaded = files.upload()

for zip_file_path in uploaded.keys():
    extract_folder = os.path.splitext(zip_file_path)[0]
    os.makedirs(extract_folder, exist_ok=True)
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
    print(f"✅ Dataset extracted to: {extract_folder}")
    dataset_path = extract_folder

Please upload your dataset ZIP file:


Saving segmentation.zip to segmentation.zip
✅ Dataset extracted to: segmentation


In [ ]:
import os

dataset_path = "/path/to/your/extracted/dataset"  # <-- update this

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{sub_indent}{f}")


In [ ]:
# ========================================================================
# CELL 1: Imports and Configuration
# ========================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from tqdm import tqdm
import random
import warnings
warnings.filterwarnings("ignore")

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF

# Segmentation Models PyTorch
!pip install -q segmentation-models-pytorch
import segmentation_models_pytorch as smp

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


Using device: cuda


In [ ]:
# ========================================================================
# CELL 2: Dataset and DataLoader
# ========================================================================

class RidgeSegmentationDataset(Dataset):
    """
    PyTorch Dataset for Ridge Segmentation (Neo & RetCam).
    """
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        # Load image and mask
        image = plt.imread(img_path)
        mask = plt.imread(mask_path)

        # Convert grayscale masks to binary if needed
        if len(mask.shape) == 3:
            mask = mask[:, :, 0]  # take first channel if RGB
        mask = (mask > 0).astype(np.float32)

        # Convert to float32 and normalize image
        image = image.astype(np.float32) / 255.0
        mask = np.expand_dims(mask, axis=0)  # [1,H,W] for SMP

        # Apply transforms
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented['image'], augmented['mask']

        # Convert to tensor
        image = torch.tensor(image).permute(2,0,1).float()  # [C,H,W]
        mask = torch.tensor(mask).float()                     # [1,H,W]

        return image, mask

# ----------------------------- Utility to load paths -----------------------------
def get_image_mask_paths(base_dir):
    neo_imgs = sorted(glob(os.path.join(base_dir, "Neo_Ridge_images", "*.*")))
    neo_masks = sorted(glob(os.path.join(base_dir, "Neo_Ridge_masks", "*.*")))
    retcam_imgs = sorted(glob(os.path.join(base_dir, "RetCam_Ridge_images", "*.*")))
    retcam_masks = sorted(glob(os.path.join(base_dir, "RetCam_Ridge_masks", "*.*")))

    all_images = neo_imgs + retcam_imgs
    all_masks = neo_masks + retcam_masks
    return all_images, all_masks

# ----------------------------- Example usage -----------------------------
BASE_DIR = "/content/segmentation/epics2/HVDROPDB-RIDGE"
image_paths, mask_paths = get_image_mask_paths(BASE_DIR)
print(f"Total images: {len(image_paths)}, Total masks: {len(mask_paths)}")

# Train/Val Split
from sklearn.model_selection import train_test_split
train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    image_paths, mask_paths, test_size=0.2, random_state=SEED
)

# DataLoader
train_dataset = RidgeSegmentationDataset(train_imgs, train_masks)
val_dataset = RidgeSegmentationDataset(val_imgs, val_masks)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print("✓ DataLoaders ready!")


Total images: 100, Total masks: 100
✓ DataLoaders ready!


In [ ]:
# ========================================================================
# CELL 3: Ridge Segmentation Model (U-Net with ResNet34 backbone)
# ========================================================================

import segmentation_models_pytorch as smp

# Define the model
model = smp.Unet(
    encoder_name="resnet34",        # Encoder: ResNet34
    encoder_weights="imagenet",     # Pretrained on ImageNet
    in_channels=3,                  # RGB input
    classes=1,                      # Binary segmentation (ridge mask)
    activation=None                 # We'll apply BCE/Dice separately
)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("✓ Ridge segmentation model initialized:")
print(model)


✓ Ridge segmentation model initialized:
Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps

In [ ]:
# ========================================================================
# CELL 4: Loss Function and Optimizer (Corrected)
# ========================================================================

# DiceLoss for segmentation overlap + BCEWithLogits for pixel-wise classification
dice_loss = smp.losses.DiceLoss(mode='binary')      # SMP DiceLoss
bce_loss = nn.BCEWithLogitsLoss()                  # BCE for binary mask

# Combined loss function
def combined_loss(preds, targets):
    return dice_loss(preds, targets) + bce_loss(preds, targets)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Learning rate scheduler (remove verbose)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

print("✓ Loss function, optimizer, and scheduler ready!")


✓ Loss function, optimizer, and scheduler ready!


In [ ]:
import torch
from torch.utils.data import Dataset
import cv2
import numpy as np

class RidgeSegDataset(Dataset):
    """Segmentation dataset with resizing and normalization."""
    def __init__(self, image_paths, mask_paths, img_size=224, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.img_size = img_size
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image and mask
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)

        # Resize both to fixed size
        img = cv2.resize(img, (self.img_size, self.img_size))
        mask = cv2.resize(mask, (self.img_size, self.img_size))

        # Normalize image
        img = img / 255.0
        mask = mask / 255.0
        mask = np.expand_dims(mask, axis=0)  # (1,H,W)

        # Convert to torch tensors
        img = torch.tensor(img, dtype=torch.float32).permute(2,0,1)  # (C,H,W)
        mask = torch.tensor(mask, dtype=torch.float32)

        if self.transform:
            img = self.transform(img)

        return img, mask


In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    """Two consecutive conv layers with BatchNorm + ReLU"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        self.dconv_down1 = DoubleConv(in_ch, 64)
        self.dconv_down2 = DoubleConv(64, 128)
        self.dconv_down3 = DoubleConv(128, 256)
        self.dconv_down4 = DoubleConv(256, 512)

        self.maxpool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        self.dconv_up3 = DoubleConv(256 + 512, 256)
        self.dconv_up2 = DoubleConv(128 + 256, 128)
        self.dconv_up1 = DoubleConv(128 + 64, 64)

        self.conv_last = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        # Encoder
        conv1 = self.dconv_down1(x)
        conv2 = self.dconv_down2(self.maxpool(conv1))
        conv3 = self.dconv_down3(self.maxpool(conv2))
        conv4 = self.dconv_down4(self.maxpool(conv3))

        # Decoder
        x = self.upsample(conv4)
        x = torch.cat([x, conv3], dim=1)
        x = self.dconv_up3(x)

        x = self.upsample(x)
        x = torch.cat([x, conv2], dim=1)
        x = self.dconv_up2(x)

        x = self.upsample(x)
        x = torch.cat([x, conv1], dim=1)
        x = self.dconv_up1(x)

        out = self.conv_last(x)
        out = torch.sigmoid(out)
        return out


In [ ]:
import os
from sklearn.model_selection import train_test_split

# Base paths
base_path = "segmentation/epics2/HVDROPDB-RIDGE"

# Neo
neo_imgs = sorted([os.path.join(base_path, "Neo_Ridge_images", f)
                   for f in os.listdir(os.path.join(base_path, "Neo_Ridge_images"))
                   if f.lower().endswith((".png", ".jpg", ".jpeg"))])

neo_masks = sorted([os.path.join(base_path, "Neo_Ridge_masks", f)
                    for f in os.listdir(os.path.join(base_path, "Neo_Ridge_masks"))
                    if f.lower().endswith((".png", ".jpg", ".jpeg"))])

# RetCam
retcam_imgs = sorted([os.path.join(base_path, "RetCam_Ridge_images", f)
                      for f in os.listdir(os.path.join(base_path, "RetCam_Ridge_images"))
                      if f.lower().endswith((".png", ".jpg", ".jpeg"))])

retcam_masks = sorted([os.path.join(base_path, "RetCam_Ridge_masks", f)
                       for f in os.listdir(os.path.join(base_path, "RetCam_Ridge_masks"))
                       if f.lower().endswith((".png", ".jpg", ".jpeg"))])

# Combine all
all_images = neo_imgs + retcam_imgs
all_masks = neo_masks + retcam_masks

# Train/Val split
train_image_paths, val_image_paths, train_mask_paths, val_mask_paths = train_test_split(
    all_images, all_masks, test_size=0.2, random_state=42
)

print(f"Training samples: {len(train_image_paths)}, Validation samples: {len(val_image_paths)}")


Training samples: 80, Validation samples: 20


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(in_ch=3, out_ch=1).to(device)

# Dataset & Loader
train_dataset = RidgeSegDataset(train_image_paths, train_mask_paths, img_size=224)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

val_dataset = RidgeSegDataset(val_image_paths, val_mask_paths, img_size=224)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

# Loss & Optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# Training
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for imgs, masks in train_loader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            val_loss += loss.item()
    val_loss /= len(val_loader)

    scheduler.step(val_loss)
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")


Epoch [1/50], Train Loss: 0.4312, Val Loss: 0.5714
Epoch [2/50], Train Loss: 0.3324, Val Loss: 0.3611
Epoch [3/50], Train Loss: 0.3063, Val Loss: 0.3108
Epoch [4/50], Train Loss: 0.2909, Val Loss: 0.2936
Epoch [5/50], Train Loss: 0.2763, Val Loss: 0.2755
Epoch [6/50], Train Loss: 0.2658, Val Loss: 0.2600
Epoch [7/50], Train Loss: 0.2583, Val Loss: 0.2591
Epoch [8/50], Train Loss: 0.2476, Val Loss: 0.2902
Epoch [9/50], Train Loss: 0.2428, Val Loss: 0.2371
Epoch [10/50], Train Loss: 0.2298, Val Loss: 0.2218
Epoch [11/50], Train Loss: 0.2202, Val Loss: 0.2210
Epoch [12/50], Train Loss: 0.2112, Val Loss: 0.2042
Epoch [13/50], Train Loss: 0.2033, Val Loss: 0.2081
Epoch [14/50], Train Loss: 0.1966, Val Loss: 0.1955
Epoch [15/50], Train Loss: 0.1880, Val Loss: 0.1836
Epoch [16/50], Train Loss: 0.1814, Val Loss: 0.1819
Epoch [17/50], Train Loss: 0.1751, Val Loss: 0.1829
Epoch [18/50], Train Loss: 0.1716, Val Loss: 0.1770
Epoch [19/50], Train Loss: 0.1637, Val Loss: 0.1659
Epoch [20/50], Train 

In [ ]:
# ------------------ Final Evaluation ------------------
model.eval()
dice_scores = []
ious = []
pixel_accuracies = []

with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc="Evaluating"):
        imgs = imgs.to(device)
        masks = masks.to(device)
        outputs = model(imgs)
        preds = (outputs > 0.5).float()

        # Dice score
        intersection = (preds * masks).sum(dim=(1,2,3))
        union = preds.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3))
        dice = (2. * intersection + 1e-8) / (union + 1e-8)
        dice_scores.extend(dice.cpu().numpy())

        # IoU
        iou = (intersection + 1e-8) / (union - intersection + 1e-8)
        ious.extend(iou.cpu().numpy())

        # Pixel Accuracy
        correct = (preds == masks).float().sum()
        total = torch.numel(preds)
        pixel_accuracies.append((correct / total).item())

# Compute final metrics
final_dice = np.mean(dice_scores)
final_iou = np.mean(ious)
final_pixel_acc = np.mean(pixel_accuracies)

print(f"\n✅ Final Evaluation on Validation Set:")
print(f"Dice Score     : {final_dice:.4f}")
print(f"IoU            : {final_iou:.4f}")
print(f"Pixel Accuracy : {final_pixel_acc:.4f}")


Evaluating: 100%|██████████| 5/5 [00:01<00:00,  2.65it/s]


✅ Final Evaluation on Validation Set:
Dice Score     : 0.5446
IoU            : 0.3928
Pixel Accuracy : 0.9758


In [ ]:
# ------------------ Save the Trained Model ------------------
model_path = "/content/segment.pth"  # update path as needed
torch.save(model.state_dict(), model_path)
print(f"✅ Model saved successfully at: {model_path}")


✅ Model saved successfully at: /content/segment.pth
